# Test 01: R-Only Installation

Integration test for the `sfnb_multilang` toolkit -- R language only.

**What this tests:**
1. Toolkit installation via pip
2. R runtime installation (micromamba + conda-forge)
3. R helper setup (`%%R` magic registration)
4. Basic R execution
5. Python <-> R data transfer
6. R package availability (tidyverse stack)
7. ADBC Snowflake driver
8. DuckDB with Snowflake extension

**Prerequisites:**
- EAI with conda-forge + CRAN access enabled on this notebook
- Or run Test 06 (EAI) first to configure network access

## 1. Install the Toolkit

In [ ]:
# Install from local source (notebooks/ is inside the package dir)
!pip install -q ..

## 2. Install R via Toolkit

This replaces `!bash setup_r_environment.sh`. The toolkit:
1. Runs preflight checks (disk, network)
2. Installs micromamba (~2s)
3. Creates conda env with R + tidyverse packages (~20s)
4. Installs ADBC Snowflake driver (~2 min, Go compilation)
5. Downloads DuckDB + Snowflake extension (~10s)
6. Validates the install

**Expected time:** ~2.5 min on fresh container, ~2s on re-run (cached).

**Configuration options:** The `install()` call below uses keyword arguments
for simplicity. For more advanced configurations (custom conda/CRAN packages,
Scala/Julia, JVM options, network rules, etc.) you can pass a YAML config
file instead:

```python
install(config="configs/r_only.yaml")
```

See the preset configs in `configs/` for examples:
- `configs/default.yaml` -- all languages with full options
- `configs/r_only.yaml` -- R + ADBC + DuckDB
- `configs/rsnowflake.yaml` -- R tuned for the RSnowflake DBI package
- `configs/snowflaker.yaml` -- R tuned for the snowflakeR ML package
- `configs/r_scala.yaml` -- R + Scala combined

In [ ]:
from sfnb_multilang import install

report = install(languages=["r"], r_adbc=True, r_duckdb=True)

In [ ]:
# Verify the report
print(f"Success: {report.success}")
print(f"Env prefix: {report.env_prefix}")
print(f"Duration: {report.elapsed_seconds:.1f}s")

r_result = report.plugin_results.get("r")
if r_result:
    print(f"R version: {r_result.version}")
    print(f"R install OK: {r_result.success}")
    if r_result.warnings:
        print(f"Warnings: {r_result.warnings}")
    if r_result.errors:
        print(f"Errors: {r_result.errors}")

## 3. Set Up R Magic

Load the R helper module (deployed by the installer to the working
directory) and register the `%%R` magic.

In [ ]:
from r_helpers import setup_r_environment
setup_r_environment()

## 4. Basic R Execution

In [ ]:
%%R
cat("R version:", R.version.string, "\n")
cat("Platform:", R.version$platform, "\n")
cat("R_HOME:", R.home(), "\n")

In [ ]:
%%R
# Verify tidyverse packages loaded
library(dplyr)
library(ggplot2)
library(tidyr)

mtcars %>%
  group_by(cyl) %>%
  summarise(
    mean_mpg = mean(mpg),
    mean_hp = mean(hp),
    count = n()
  )

## 5. Python <-> R Data Transfer

In [ ]:
import pandas as pd

# Create a Python DataFrame
py_df = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol"],
    "score": [95, 87, 92]
})
print("Python DataFrame:")
print(py_df)

In [ ]:
%%R -i py_df -o r_result
# Receive Python DF, transform in R, send back
r_result <- py_df %>%
  mutate(grade = ifelse(score >= 90, "A", "B")) %>%
  arrange(desc(score))
r_result

In [ ]:
# Verify we got the R result back in Python
print("\nBack in Python:")
print(r_result)
assert "grade" in r_result.columns, "FAIL: grade column missing"
assert len(r_result) == 3, "FAIL: wrong row count"
print("\nPASS: Python <-> R data transfer works")

## 6. R Package Availability

Verify the standard conda packages are importable.

In [ ]:
%%R
required_packages <- c("dplyr", "ggplot2", "tidyr", "dbplyr", "httr2", "reticulate")

for (pkg in required_packages) {
  ok <- requireNamespace(pkg, quietly = TRUE)
  status <- if (ok) "OK" else "MISSING"
  cat(sprintf("  %-15s %s\n", pkg, status))
  if (!ok) stop(paste("FAIL: package", pkg, "not found"))
}
cat("\nPASS: All required R packages available\n")

## 7. Snowflake Connectivity

Test that R can query Snowflake via the Python session.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

sf_df = session.sql("SELECT CURRENT_ACCOUNT() AS acct, CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS wh").to_pandas()
print(sf_df)

In [ ]:
%%R -i sf_df
cat("Snowflake connection (via Python proxy):\n")
print(sf_df)
cat("\nPASS: R can receive Snowflake data\n")

## 8. ADBC Add-on Test

Verify the ADBC Snowflake driver installed correctly (~2 min install).
This requires network access to r-multiverse, Go proxy, and googleapis.

In [ ]:
%%R
library(adbcdrivermanager)
library(adbcsnowflake)
cat("PASS: ADBC Snowflake driver loaded\n")

## 9. DuckDB Add-on Test

Verify DuckDB with the Snowflake extension installed correctly (~10s install).
Extensions are downloaded over HTTPS via Python and placed in R's DuckDB directory.

In [ ]:
%%R
library(DBI)
library(duckdb)
con <- dbConnect(duckdb::duckdb(), dbdir = ":memory:")
dbExecute(con, "SET autoinstall_known_extensions=false")
dbExecute(con, "LOAD httpfs")
dbExecute(con, "LOAD snowflake")
cat("PASS: DuckDB Snowflake extension loaded\n")
dbDisconnect(con)

## Test Summary

If all cells above ran without errors, the R-only integration test passes.

| Test | Expected |
|---|---|
| Toolkit install | pip succeeds |
| R install | report.success == True |
| R magic | `%%R` cells execute |
| Data transfer | Python DF -> R -> Python round-trip |
| Packages | All tidyverse packages loadable |
| Snowflake | R receives Snowflake data via Python proxy |
| ADBC | adbcdrivermanager + adbcsnowflake load |
| DuckDB | DuckDB Snowflake extension loads |